# Battle Lab · LIGHT M-C v2 — PPO visible y reanudable 🐉⚔️

Piloto corregido de `BATTLE-LAB-MC-TRAIN-001`. Reutiliza el censo, el split 145/36 y el baseline BC M-A/M-B. No ejecuta la pre-evaluación interna bloqueante de VGC-Bench. Extiende de forma **append-only** los catálogos M-C para conservar los IDs aprendidos por el checkpoint público, muestra fase/barra/ETA desde el inicio y reanuda desde el último checkpoint persistido en Drive.


In [ ]:
TOTAL_STEPS = 196_608  # @param {type:"integer"}
NUM_ENVS = 2  # @param {type:"integer"}
CHECKPOINT_EVERY = 24_576  # @param {type:"integer"}
DEVICE = "auto"  # @param ["auto", "cuda", "cpu"]
SEED = 260913
PORT = 8000
PKMN_REPOSITORY = "https://github.com/Iesyo/pkmn.git"
PKMN_REF = "main"
VGC_BENCH_REPOSITORY = "https://github.com/cameronangliss/vgc-bench.git"
VGC_BENCH_COMMIT = "d79f9532947ac114dce1dda2456a590afcd375b2"
NODE_VERSION = "24.21.0"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/Colabs/LikeNoOneEverWas/BattleLab/MC-Training")
DATA_ROOT = DRIVE_ROOT / "data"
SPLIT_ROOT = DATA_ROOT / "team-splits" / f"seed-{SEED}"
TRAIN_TEAM_DIR = SPLIT_ROOT / "train"
HOLDOUT_TEAM_DIR = SPLIT_ROOT / "holdout"
OUTPUT_ROOT = DRIVE_ROOT / "training"
LOG_ROOT = DRIVE_ROOT / "logs"
PKMN_ROOT = Path("/content/pkmn")
VGC_BENCH_ROOT = Path("/content/vgc-bench")
SHOWDOWN_ROOT = Path("/content/battle-lab-runtime/pokemon-showdown")
for p in (OUTPUT_ROOT, LOG_ROOT):
    p.mkdir(parents=True, exist_ok=True)
print("📁", DRIVE_ROOT)


In [ ]:
import json, os, subprocess, sys, time

def run(command, *, cwd=None, label=None):
    if label:
        print(f"\n▶ {label}", flush=True)
    subprocess.run([str(x) for x in command], cwd=cwd, check=True)

if not (PKMN_ROOT / ".git").is_dir():
    run(["git", "clone", "--filter=blob:none", PKMN_REPOSITORY, str(PKMN_ROOT)], label="Clonando pkmn")
run(["git", "fetch", "origin", PKMN_REF], cwd=PKMN_ROOT, label="Actualizando pkmn")
run(["git", "checkout", "--force", "FETCH_HEAD"], cwd=PKMN_ROOT, label="Fijando pkmn")
pkmn_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PKMN_ROOT, text=True).strip()
print("pkmn SHA:", pkmn_sha)

node_major = int(subprocess.check_output(["node", "-p", "process.versions.node.split('.')[0]"], text=True).strip())
if node_major < 24:
    run(["npm", "install", "-g", "n"], label="Instalando selector Node")
    run(["n", NODE_VERSION], label=f"Node {NODE_VERSION}")
    os.environ["PATH"] = "/usr/local/bin:" + os.environ["PATH"]

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PKMN_ROOT / "battle_lab" / "requirements-phase1.txt")], label="Dependencias Battle Lab")
run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PKMN_ROOT / "battle_lab" / "requirements-mc-train.txt")], label="Dependencias entrenamiento")

sys.path.insert(0, str(PKMN_ROOT))
from battle_lab.showdown_smoke import DEFAULT_SHOWDOWN_REPOSITORY, ensure_showdown_checkout, install_runtime_config, read_showdown_commit
showdown_commit = read_showdown_commit()
ensure_showdown_checkout(checkout=SHOWDOWN_ROOT, repository=DEFAULT_SHOWDOWN_REPOSITORY, commit=showdown_commit, logs_dir=LOG_ROOT)
install_runtime_config(SHOWDOWN_ROOT)
from battle_lab.vgc_bench_battle import ensure_vgc_bench_checkout
ensure_vgc_bench_checkout(checkout=VGC_BENCH_ROOT, repository=VGC_BENCH_REPOSITORY, commit=VGC_BENCH_COMMIT)
print("Showdown SHA:", showdown_commit)
print("VGC-Bench SHA:", VGC_BENCH_COMMIT)


In [ ]:
split_manifest_path = SPLIT_ROOT / "split_manifest.json"
if not split_manifest_path.exists():
    raise RuntimeError("Falta split_manifest.json; ejecuta primero el censo M-C.")
split_manifest = json.loads(split_manifest_path.read_text())
if int(split_manifest.get("signatureLeakage", -1)) != 0:
    raise RuntimeError(f"Fuga train/holdout detectada: {split_manifest}")
if int(split_manifest.get("trainTeams", 0)) < 100 or int(split_manifest.get("holdoutTeams", 0)) < 20:
    raise RuntimeError(f"Split incompleto: {split_manifest}")
print(f"✅ Split verificado: train={split_manifest['trainTeams']} · holdout={split_manifest['holdoutTeams']} · fuga=0")


In [ ]:
from battle_lab.showdown_smoke import running_showdown
print("🚦 Iniciando LIGHT v2. Debe imprimir fase y progreso inmediatamente.", flush=True)
with running_showdown(SHOWDOWN_ROOT, PORT, LOG_ROOT / "mc-light-showdown.log"):
    run([
        sys.executable, "-m", "battle_lab.mc_rl_light_v2",
        "--vgc-bench", str(VGC_BENCH_ROOT),
        "--output-root", str(OUTPUT_ROOT),
        "--team-dir", str(TRAIN_TEAM_DIR),
        "--port", str(PORT),
        "--device", DEVICE,
        "--seed", str(SEED),
        "--total-steps", str(TOTAL_STEPS),
        "--num-envs", str(NUM_ENVS),
        "--checkpoint-every", str(CHECKPOINT_EVERY),
    ], cwd=PKMN_ROOT, label="PPO/self-play M-C v2 visible")


In [ ]:
summary_path = OUTPUT_ROOT / "rl" / "light" / f"seed{SEED}" / "summary.json"
status_path = OUTPUT_ROOT / "rl" / "light" / f"seed{SEED}" / "status.json"
if not summary_path.exists():
    status = json.loads(status_path.read_text()) if status_path.exists() else {}
    raise RuntimeError(f"LIGHT no terminó. Estado: {status}")
summary = json.loads(summary_path.read_text())
print("\n===== BATTLE LAB LIGHT M-C v2 =====")
print("Estado:", summary.get("state"))
print("Inicio/reanudación:", f"{summary.get('startStep', 0):,}")
print("Paso final:", f"{summary.get('finalStep', 0):,}", "/", f"{summary.get('totalSteps', TOTAL_STEPS):,}")
print("Checkpoint:", summary.get("finalCheckpoint"))
print("SHA256:", summary.get("finalCheckpointSha256"))
print("Segundos:", summary.get("seconds"))
print("\n✅ Siguiente paso: benchmark contra el baseline usando únicamente los 36 equipos holdout.")
